# Geographical Context Neural Network



This notebook trains a neural network for Surface Water Fraction estimation using geographical context. Each sample is built from one central grid cell plus its neighbouring cells inside a fixed spatial window, so the model can learn local spatial structure instead of treating each point independently.


## Setup



Import the tools needed to build neighbourhood-aware samples, train the model, and evaluate the predictions.


In [3]:
import matplotlib.pyplot as plt

import numpy as np

import pandas as pd



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import train_test_split

from sklearn.neural_network import MLPRegressor

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler



RANDOM_STATE = 42

TARGET_COL = "fwns"

CONTEXT_RADIUS = 1

TEST_SIZE = 0.2

np.random.seed(RANDOM_STATE)


In [4]:
FEATURES = [

    "tbtoa19H",

    "tbtoa37H",

    "tbtoa19V",

    "tbtoa37V",

    "tran19H",

    "tran37H",

    "tran19V",

    "tran37V",

    "surtep_ERA5",

    "vsm",

    "VOD",

    "Tmn",

    "PWV",

    "VPD",

]


In [5]:
def make_center_key(df_in):

    return (

        df_in["day"].astype(str)

        + "|"

        + df_in["lat_idx"].astype(str)

        + "|"

        + df_in["lon_idx"].astype(str)

    )





def add_grid_indices(df_in, lat_col="latitude_grid", lon_col="longitude_grid"):

    df_out = df_in.copy()

    lat_values = np.sort(df_out[lat_col].unique())

    lon_values = np.sort(df_out[lon_col].unique())



    lat_map = {value: idx for idx, value in enumerate(lat_values)}

    lon_map = {value: idx for idx, value in enumerate(lon_values)}



    df_out["lat_idx"] = df_out[lat_col].map(lat_map).astype(int)

    df_out["lon_idx"] = df_out[lon_col].map(lon_map).astype(int)

    df_out["center_key"] = make_center_key(df_out)

    return df_out





def build_context_dataset(df_source, feature_cols, radius=1, target_col=TARGET_COL):

    base_cols = ["day", "latitude_grid", "longitude_grid", "lat_idx", "lon_idx", target_col, "center_key"]

    source = df_source[base_cols + feature_cols].copy()

    centers = source[base_cols].copy()

    merged = centers.copy()

    lookup = source[["day", "lat_idx", "lon_idx", *feature_cols]].copy()

    context_cols = []



    for dy in range(-radius, radius + 1):

        for dx in range(-radius, radius + 1):

            shifted = lookup.copy()

            shifted["lat_idx"] = shifted["lat_idx"] - dy

            shifted["lon_idx"] = shifted["lon_idx"] - dx



            renamed = {

                feature: f"{feature}__dy{dy:+d}_dx{dx:+d}"

                for feature in feature_cols

            }

            shifted = shifted.rename(columns=renamed)

            context_cols.extend(renamed.values())



            merged = merged.merge(

                shifted,

                on=["day", "lat_idx", "lon_idx"],

                how="left",

            )



    merged = merged.dropna(subset=context_cols + [target_col]).reset_index(drop=True)

    return merged, context_cols





def evaluate_predictions(y_true, y_pred):

    return {

        "MAE": mean_absolute_error(y_true, y_pred),

        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),

        "R2": r2_score(y_true, y_pred),

    }


## Data Preparation



Load the dataset, convert the latitude and longitude coordinates into grid indices, and expand each central point into a neighbourhood patch.


In [6]:
df_raw = pd.read_parquet("data/windsat_2017")

df_grid = add_grid_indices(df_raw)

context_df, context_cols = build_context_dataset(

    df_grid,

    feature_cols=FEATURES,

    radius=CONTEXT_RADIUS,

)



train_keys, test_keys = train_test_split(

    context_df["center_key"].unique(),

    test_size=TEST_SIZE,

    random_state=RANDOM_STATE,

    shuffle=True,

)



train_mask = context_df["center_key"].isin(train_keys)

test_mask = context_df["center_key"].isin(test_keys)



X_train = context_df.loc[train_mask, context_cols]

y_train = context_df.loc[train_mask, TARGET_COL]

X_test = context_df.loc[test_mask, context_cols]

y_test = context_df.loc[test_mask, TARGET_COL]



print(f"Original rows: {len(df_grid):,}")

print(f"Rows with full context: {len(context_df):,}")

print(f"Input features per sample: {len(context_cols)}")

print(f"Training rows: {len(X_train):,}")

print(f"Test rows: {len(X_test):,}")


Original rows: 13,225,757
Rows with full context: 9,689,782
Input features per sample: 126
Training rows: 7,751,825
Test rows: 1,937,957


## Training And Evaluation



Train a multilayer perceptron on the context-expanded samples and evaluate it on a held-out test set.


In [ ]:
model = Pipeline([

    ("scaler", StandardScaler()),

    (

        "mlp",

        MLPRegressor(

            hidden_layer_sizes=(256, 128),

            activation="relu",

            solver="adam",

            alpha=1e-4,

            batch_size=1024,

            learning_rate_init=1e-3,

            max_iter=120,

            early_stopping=True,

            validation_fraction=0.1,

            n_iter_no_change=10,

            random_state=RANDOM_STATE,

        ),

    ),

])



model.fit(X_train, y_train)



y_pred_train = model.predict(X_train)

y_pred_test = model.predict(X_test)



train_metrics = evaluate_predictions(y_train, y_pred_train)

test_metrics = evaluate_predictions(y_test, y_pred_test)



metrics_df = pd.DataFrame([

    {"split": "train", **train_metrics},

    {"split": "test", **test_metrics},

])

display(metrics_df)


c:\Users\marce\miniconda3\envs\TFM_MCD\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
results_df = pd.DataFrame({

    "y_true": y_test,

    "y_pred": y_pred_test,

})

results_df["residual"] = results_df["y_true"] - results_df["y_pred"]



display(results_df.head(10))



fig, axes = plt.subplots(1, 2, figsize=(12, 5))



axes[0].scatter(results_df["y_true"], results_df["y_pred"], s=8, alpha=0.35)

axes[0].plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)

axes[0].set_title("Test Predictions vs Ground Truth")

axes[0].set_xlabel("Observed fwns")

axes[0].set_ylabel("Predicted fwns")



axes[1].hist(results_df["residual"], bins=40)

axes[1].set_title("Residual Distribution")

axes[1].set_xlabel("y_true - y_pred")

axes[1].set_ylabel("Frequency")



plt.tight_layout()

plt.show()


## Notes



The model now uses a 3x3 neighbourhood around each grid cell, so every prediction is informed by local spatial structure. If you want a larger or smaller geographical context, change `CONTEXT_RADIUS` and rerun the notebook.
